# Chapter 13 — CLIP and Contrastive Language-Image Pretraining

Every model so far has learned about images from **labels you supplied**: ten
classes on Fashion-MNIST, three on beans, one integer per picture. That is a
remarkably thin channel. A label cannot say *"a dark narrow boot"*, cannot
describe something it has no class for, and cannot be written by anyone who is
not annotating your dataset.

The web, meanwhile, is full of images that already come with text attached:
alt tags, captions, filenames. **CLIP** (Radford et al., 2021) is the
observation that this is enough to learn from, if you pick the right objective:
don't predict the caption, just learn to *match* images to their captions and
away from everyone else's. The supervision is free, and it arrives in natural
language rather than in a fixed vocabulary of class indices.

You have already built both halves of the model. The image tower is Chapter 11's
ViT with the classifier head removed; the text tower is Chapter 10's encoder
block over tokenized text. The only new thing is what joins them:

$$ \mathcal{L} = \tfrac{1}{2}\Big[\text{CE}\big(\tfrac{1}{\tau} Z_I Z_T^\top,\; \mathrm{I}\big) + \text{CE}\big(\tfrac{1}{\tau} Z_T Z_I^\top,\; \mathrm{I}\big)\Big] $$

One matrix of similarities, and a loss that says *the diagonal is correct*.

| Module | What you build | Dataset (Hugging Face) |
|---|---|---|
| 1 | Image–text pairs, and why a caption carries more than a label | `zalando-datasets/fashion_mnist` |
| 2 | Two encoders projecting into **one shared space**, and the contrastive loss | — |
| 3 | Training, and reading the similarity grid | — |
| 4 | **Zero-shot classification**, against a supervised baseline | — |
| 5 | **Compositional** prompts: querying attributes nobody labelled | — |
| 6 | Two measurements that complicate the story: batch size, and the **modality gap** | — |

**How each concept is presented**, the same three passes as earlier chapters:

> 🧠 **The intuition:** the idea in plain language, no symbols.
> 📐 **The math:** the same idea written precisely, so you can read papers.
> 💻 **The code:** the same idea again, executable, in the cell that follows.

**Runtime:** ~3 minutes on a GPU. The main training run is about 11 seconds;
most of the time goes to Module 6, which trains six more models for the two
sweeps. Knobs are marked `# <- knob`. Fashion-MNIST and the BERT tokenizer are
already cached from earlier chapters.


In [ ]:
import math, time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoTokenizer

torch.manual_seed(0)
np.random.seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
plt.rcParams["figure.figsize"] = (7, 4.5)

print("torch", torch.__version__, "| device:", device)

---
# Module 1 — Pairs, Not Labels

## 1.1 What a caption carries that a label does not

🧠 **The intuition.** A label is a pointer into a list you fixed in advance.
Ask a label-trained classifier about a *dark narrow boot* and it has no way to
represent the question, since "dark" and "narrow" were never in its output
layer, and adding them means relabelling the dataset.

A caption is a sentence. It composes. If a model learns what "dark" means from
photographs of dark trousers and dark bags, it can apply that word to a boot it
has never been told is dark. That compositionality is the whole reason to train
on language instead of on class indices, and Module 5 measures whether we got it.

📐 **The setup.** We need pairs $(x_i, t_i)$, an image and a matching string.
Real CLIP scraped 400 million of them. We are going to *construct* ours from
Fashion-MNIST, and it is worth being explicit about the compromise:

- **What is honest about it:** the caption for each image is built from
  attributes **measured off that specific image** (its mean brightness and how
  many columns it occupies) rather than from anything the model is separately
  told. Those attributes are never supplied as training labels. If the model ends
  up able to answer "which of these is narrow?", it learned that from language.
- **What is not:** the vocabulary is tiny and the grammar is a template. A real
  caption corpus is far messier, and messiness is most of what makes CLIP hard.

💻 **The code.** Load the images first.


In [ ]:
fashion = load_dataset("zalando-datasets/fashion_mnist")
CLASS_NAMES = [c.replace(" - ", "-").replace(" / ", "/") for c in fashion["train"].features["label"].names]
print(CLASS_NAMES)

N_TRAIN, N_TEST = 20_000, 5_000            # <- knob: up to 60_000 / 10_000


def to_arrays(split, n):
    ds = split.shuffle(seed=42).select(range(n))
    X = np.stack([np.array(im) for im in ds["image"]]).astype("float32") / 255.0
    return X, np.array(ds["label"])


Xtr, ytr = to_arrays(fashion["train"], N_TRAIN)
Xte, yte = to_arrays(fashion["test"], N_TEST)
print("train:", Xtr.shape, "| test:", Xte.shape)

## 1.2 Captions built from measured attributes

💻 **The code.** Two attributes, both computed from the pixels: mean brightness
(*dark* / *bright*) and how many columns the garment occupies (*narrow* /
*wide*). Split each at the training-set median, and paste the words into a
template alongside the class name.

Note the thresholds come from the **training** set and are then applied to test
images unchanged, the same discipline as fitting a `StandardScaler` on train
only, back in Chapter 1.


In [ ]:
SHADE = ["dark", "bright"]
WIDTH = ["narrow", "wide"]


def attributes(X):
    ink = X.reshape(len(X), -1).mean(1)                 # mean brightness
    cols = (X.max(axis=1) > 0.25).sum(axis=1)           # occupied columns = width
    return ink, cols


ink_tr, col_tr = attributes(Xtr)
ink_thr, col_thr = np.median(ink_tr), np.median(col_tr)     # thresholds from TRAIN only


def captions(X, y):
    ink, col = attributes(X)
    s = (ink > ink_thr).astype(int)
    w = (col > col_thr).astype(int)
    text = [f"a {SHADE[si]} {WIDTH[wi]} photo of a {CLASS_NAMES[yi]}"
            for si, wi, yi in zip(s, w, y)]
    return text, s, w


cap_tr, s_tr, w_tr = captions(Xtr, ytr)
cap_te, s_te, w_te = captions(Xte, yte)

for c in cap_tr[:4]:
    print(" ", c)
print(f"\n{len(set(cap_tr))} distinct captions out of 10 x 2 x 2 = 40 possible "
      f"({40 - len(set(cap_tr))} combination(s) never occur)")

tok = AutoTokenizer.from_pretrained("bert-base-uncased")
MAX_LEN = 16                                            # <- knob
enc_tr = tok(cap_tr, padding="max_length", truncation=True, max_length=MAX_LEN, return_tensors="pt")
print("token ids:", tuple(enc_tr["input_ids"].shape))

---
# Module 2 — Two Encoders, One Space

## 2.1 The architecture

🧠 **The intuition.** Images and text are not remotely the same kind of object,
so there is no way to compare them directly. CLIP's answer is almost crude:
build a separate encoder for each, and end both with a linear layer into the
**same** $d$-dimensional space. Then "how well does this caption describe this
image" is a dot product.

Nothing forces the two spaces to align at initialization: they are random
projections of unrelated networks. The alignment is *entirely* the loss's doing,
which is why Module 2.3 is worth running before training.

📐 **The math.** With image encoder $f_I$ and text encoder $f_T$:

$$ z_I = \frac{W_I\, f_I(x)}{\lVert W_I\, f_I(x) \rVert}, \qquad z_T = \frac{W_T\, f_T(t)}{\lVert W_T\, f_T(t) \rVert} $$

Both are **L2-normalized**, so every embedding lives on the unit sphere and the
dot product $z_I \cdot z_T$ *is* the cosine similarity, bounded in $[-1, 1]$.
That bound is what makes the temperature in the next section meaningful.

💻 **The code.** The encoder block is Chapter 10's, verbatim in structure:
pre-LN, multi-head attention, an MLP, both wrapped in residuals. The image tower
is Chapter 11's ViT with its classifier head replaced by a projection; the text
tower is the same block over token embeddings, reading out at the last real
token.


In [ ]:
D_EMB = 128                                             # <- knob: shared-space width


class Block(nn.Module):
    '''Chapter 10's pre-LN encoder block, using torch's fused attention.'''

    def __init__(self, d, h, ratio=4):
        super().__init__()
        self.n1, self.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.attn = nn.MultiheadAttention(d, h, batch_first=True)
        self.mlp = nn.Sequential(nn.Linear(d, ratio * d), nn.GELU(), nn.Linear(ratio * d, d))

    def forward(self, x, pad_mask=None):
        a = self.n1(x)
        x = x + self.attn(a, a, a, key_padding_mask=pad_mask, need_weights=False)[0]
        return x + self.mlp(self.n2(x))


class ImageEncoder(nn.Module):
    '''Chapter 11's ViT with the classifier head swapped for a projection.'''

    def __init__(self, d=128, patch=4, depth=4, heads=4, out_dim=D_EMB):
        super().__init__()
        self.patch = nn.Conv2d(1, d, kernel_size=patch, stride=patch)   # the Conv2d trick
        n = (28 // patch) ** 2
        self.cls = nn.Parameter(torch.zeros(1, 1, d))
        self.pos = nn.Parameter(torch.randn(1, n + 1, d) * 0.02)
        self.blocks = nn.ModuleList([Block(d, heads) for _ in range(depth)])
        self.norm = nn.LayerNorm(d)
        self.proj = nn.Linear(d, out_dim, bias=False)                   # into the shared space

    def forward(self, x):
        x = self.patch(x).flatten(2).transpose(1, 2)
        x = torch.cat([self.cls.expand(len(x), -1, -1), x], 1) + self.pos
        for b in self.blocks:
            x = b(x)
        return self.proj(self.norm(x)[:, 0])                            # read out [CLS]


class TextEncoder(nn.Module):
    '''The same block over BERT-tokenized captions.'''

    def __init__(self, vocab, d=128, depth=4, heads=4, max_len=MAX_LEN, out_dim=D_EMB):
        super().__init__()
        self.emb = nn.Embedding(vocab, d)
        self.pos = nn.Parameter(torch.randn(1, max_len, d) * 0.02)
        self.blocks = nn.ModuleList([Block(d, heads) for _ in range(depth)])
        self.norm = nn.LayerNorm(d)
        self.proj = nn.Linear(d, out_dim, bias=False)

    def forward(self, ids, mask):
        x = self.emb(ids) + self.pos[:, : ids.shape[1]]
        pad = mask == 0                                    # don't attend to padding
        for b in self.blocks:
            x = b(x, pad_mask=pad)
        x = self.norm(x)
        eos = mask.sum(1) - 1                              # last real token = the summary slot
        return self.proj(x[torch.arange(len(x)), eos])

## 2.2 The contrastive loss

🧠 **The intuition.** Take a batch of $N$ (image, caption) pairs and compute
every image against every caption: an $N \times N$ grid of similarities. Exactly
$N$ of those cells are correct, the diagonal, and the other $N^2 - N$ are
wrong *by construction*, because they pair an image with somebody else's caption.

So the loss is just classification, twice. Read the grid **by rows**: "given this
image, which of the $N$ captions is yours?" Read it **by columns**: "given this
caption, which image?" Both are cross-entropy against the diagonal, and CLIP
averages them.

The elegance is that the negatives are free. Nobody had to annotate what a
picture *isn't*: every other item in the batch volunteers.

📐 **The math.** With $Z_I, Z_T \in \mathbb{R}^{N \times d}$ (rows unit-norm) and
a learned temperature $\tau$:

$$ S = \frac{1}{\tau}\, Z_I Z_T^\top \in \mathbb{R}^{N \times N}, \qquad
\mathcal{L} = \tfrac{1}{2}\big[\underbrace{\text{CE}(S, \mathrm{diag})}_{\text{image} \to \text{text}} + \underbrace{\text{CE}(S^\top, \mathrm{diag})}_{\text{text} \to \text{image}}\big] $$

**Why a temperature at all?** Cosine similarity is stuck in $[-1, 1]$, so raw
logits span a range of 2 and the softmax over them is almost flat, with barely
any gradient. Dividing by $\tau \approx 0.07$ stretches that to $[-14, 14]$,
which is a usable logit range. It is the same saturation argument as $\sqrt{d_k}$
in Chapter 9, Module 3.2, run in the opposite direction: there we *shrank* logits
that had grown too large, here we *grow* logits that are structurally too small.

CLIP learns $\tau$ rather than fixing it, by parameterizing $\log(1/\tau)$ so it
stays positive, the same "predict the log" trick as the VAE's `z_log_var` in
Chapter 6.

💻 **The code.**


In [ ]:
class CLIP(nn.Module):
    def __init__(self, vocab):
        super().__init__()
        self.image, self.text = ImageEncoder(), TextEncoder(vocab)
        self.logit_scale = nn.Parameter(torch.tensor(math.log(1 / 0.07)))   # learn log(1/tau)

    def encode(self, imgs, ids, mask):
        zi = F.normalize(self.image(imgs), dim=-1)      # onto the unit sphere...
        zt = F.normalize(self.text(ids, mask), dim=-1)  # ...so a dot product is a cosine
        return zi, zt

    def forward(self, imgs, ids, mask):
        zi, zt = self.encode(imgs, ids, mask)
        scale = self.logit_scale.exp().clamp(max=100)   # clamp: runaway temperature is a real failure
        return scale * zi @ zt.T, zi, zt                # (N, N) similarity grid


def clip_loss(logits):
    '''Cross-entropy against the diagonal, both ways round.'''
    tgt = torch.arange(len(logits), device=logits.device)
    return 0.5 * (F.cross_entropy(logits, tgt) + F.cross_entropy(logits.T, tgt))


img_tr = torch.from_numpy(Xtr).unsqueeze(1)             # (N, 1, 28, 28)
img_te = torch.from_numpy(Xte).unsqueeze(1)
print(f"CLIP parameters: {sum(p.numel() for p in CLIP(tok.vocab_size).parameters()):,}")

## 2.3 The grid before training

💻 **Worth one cell.** If the shared space really is unaligned at
initialization, the similarity grid should show no diagonal at all, and the loss
should sit near $\log N$, the value you get from guessing uniformly among $N$
options. Check both rather than assume them.


In [ ]:
untrained = CLIP(tok.vocab_size).to(device)
N_SHOW = 16
with torch.no_grad():
    logits0, _, _ = untrained(img_tr[:N_SHOW].to(device),
                              enc_tr["input_ids"][:N_SHOW].to(device),
                              enc_tr["attention_mask"][:N_SHOW].to(device))
    loss0 = clip_loss(logits0).item()

print(f"untrained loss: {loss0:.4f}   |   log(N) for N={N_SHOW}: {math.log(N_SHOW):.4f}")

plt.figure(figsize=(5.2, 4.4))
plt.imshow(logits0.cpu(), cmap="RdBu_r")
plt.colorbar(label="similarity / tau")
plt.xlabel("caption"); plt.ylabel("image")
plt.title("Similarity grid at initialization: no diagonal")
plt.tight_layout()
plt.show()

---
# Module 3 — Training

💻 **The code.** An ordinary supervised loop, which is the point: there is no
adversary, no sampling, no KL term. The only unusual line is that the *labels*
are `arange(N)`, manufactured from the batch's own ordering rather than read
from the dataset.


In [ ]:
def train(batch=256, epochs=8, lr=3e-4, seed=0, log=True):   # <- knobs
    torch.manual_seed(seed)
    model = CLIP(tok.vocab_size).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.1)
    ids, mask = enc_tr["input_ids"], enc_tr["attention_mask"]
    hist = []
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(len(img_tr))
        tot, nb = 0.0, 0
        for i in range(0, len(perm) - batch + 1, batch):
            j = perm[i : i + batch]
            logits, _, _ = model(img_tr[j].to(device), ids[j].to(device), mask[j].to(device))
            loss = clip_loss(logits)
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item(); nb += 1
        hist.append(tot / nb)
        if log:
            print(f"  epoch {ep+1:>2}  loss {hist[-1]:.4f}   learned tau {1/model.logit_scale.exp().item():.4f}")
    return model, hist


t0 = time.time()
model, hist = train()
print(f"\ntrained in {time.time() - t0:.1f}s")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(range(1, len(hist) + 1), hist, marker="o")
axes[0].axhline(math.log(256), color="gray", linestyle="--", label="log(N) = chance")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("contrastive loss")
axes[0].set_title("Training loss"); axes[0].grid(True); axes[0].legend()

with torch.no_grad():
    logits1, _, _ = model(img_tr[:N_SHOW].to(device),
                          enc_tr["input_ids"][:N_SHOW].to(device),
                          enc_tr["attention_mask"][:N_SHOW].to(device))
im = axes[1].imshow(logits1.cpu(), cmap="RdBu_r")
plt.colorbar(im, ax=axes[1], label="similarity / tau")
axes[1].set_xlabel("caption"); axes[1].set_ylabel("image")
axes[1].set_title("The same grid, after training")
plt.tight_layout()
plt.show()

The diagonal is now the brightest thing in the picture, and the loss sits well
under the $\log N$ chance line. Note also what the off-diagonal is *not*: it is
not uniformly cold. Images of a coat score highly against a pullover's caption,
because those genuinely are similar garments and the objective never asked the
model to pretend otherwise. Contrastive training treats every non-pair as a
negative, which is a **mild lie**: some of those "wrong" captions are nearly
right, and at scale this shows up as a real source of gradient noise.


---
# Module 4 — Zero-Shot Classification

🧠 **The intuition.** Here is the trick that made CLIP famous. We never trained
a classifier. But we can *build* one at inference time out of nothing but
sentences: write one prompt per class, encode them, and label each image with
whichever prompt it is closest to. The classifier's weights are the text
embeddings, and they are produced by typing.

That means the label set is editable after training. Add a class by writing a
sentence: no retraining, no data.

📐 **The math.** With prompts $t_1 \dots t_C$:

$$ \hat{y}(x) = \arg\max_{c}\; \frac{f_I(x)}{\lVert f_I(x)\rVert} \cdot \frac{f_T(t_c)}{\lVert f_T(t_c)\rVert} $$

Compare that to a linear classifier head $\arg\max_c\, w_c \cdot h(x)$. It is the
*same equation*, and the only difference is where $w_c$ came from. In a
supervised model it was learned from labels; here it was written in English.

💻 **The code.**


In [ ]:
@torch.no_grad()
def embed_images(m, imgs, bs=512):
    m.eval()
    return torch.cat([F.normalize(m.image(imgs[i:i+bs].to(device)), dim=-1)
                      for i in range(0, len(imgs), bs)])


@torch.no_grad()
def embed_texts(m, texts):
    m.eval()
    e = tok(texts, padding="max_length", truncation=True, max_length=MAX_LEN, return_tensors="pt")
    return F.normalize(m.text(e["input_ids"].to(device), e["attention_mask"].to(device)), dim=-1)


Zi = embed_images(model, img_te)
prompts = [f"a photo of a {c}" for c in CLASS_NAMES]
Zt = embed_texts(model, prompts)

zs_pred = (Zi @ Zt.T).argmax(1).cpu().numpy()
zs_acc = (zs_pred == yte).mean()
print(f"zero-shot accuracy from 10 typed sentences: {zs_acc:.4f}")

## 4.1 The baseline that keeps it honest

💻 A zero-shot number means nothing on its own. Train the **same image encoder**,
same size, same epochs, same optimizer, but with an ordinary 10-way
cross-entropy on the labels. That is the model CLIP has to justify itself
against.


In [ ]:
class Supervised(nn.Module):
    '''Identical image tower, ordinary classifier head.'''

    def __init__(self):
        super().__init__()
        self.enc = ImageEncoder(out_dim=10)

    def forward(self, x):
        return self.enc(x)


def train_supervised(epochs=8, batch=256, lr=3e-4, seed=0):
    torch.manual_seed(seed)
    m = Supervised().to(device)
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=0.1)
    yt = torch.from_numpy(ytr).long()
    for _ in range(epochs):
        m.train()
        perm = torch.randperm(len(img_tr))
        for i in range(0, len(perm) - batch + 1, batch):
            j = perm[i:i+batch]
            loss = F.cross_entropy(m(img_tr[j].to(device)), yt[j].to(device))
            opt.zero_grad(); loss.backward(); opt.step()
    m.eval()
    with torch.no_grad():
        p = torch.cat([m(img_te[i:i+512].to(device)).argmax(1).cpu()
                       for i in range(0, len(img_te), 512)])
    return (p.numpy() == yte).mean()


sup_acc = train_supervised()
print(f"supervised baseline (same encoder, real labels): {sup_acc:.4f}")
print(f"zero-shot CLIP:                                  {zs_acc:.4f}")
print(f"\nCLIP retains {zs_acc / sup_acc:.1%} of supervised accuracy having never seen a label.")

## 4.2 Reading the result

**The supervised model wins, and it should.** It was handed exactly the ten
categories it is scored on, and trained to separate precisely those. CLIP had to
infer the same ten distinctions from sentences, through a text encoder that also
had to learn English word by word from 20,000 examples. Losing a few points to a
model with a strictly easier job is the expected outcome, not a defect.

**The interesting comparison is not accuracy but what each model can be asked
for.** The supervised model has exactly ten questions it can answer, forever.
CLIP's classifier is a list of strings, and the next module changes that list to
ask something the label set cannot express at all.

That is the real CLIP story too, and it is worth being precise about the scale
gap: the paper's zero-shot ImageNet result (76.2%, matching a supervised
ResNet-50) came from **400 million** pairs. At 20,000 pairs of templated
captions we are four orders of magnitude short, and the gap above is what that
looks like.


---
# Module 5 — Compositional Prompts: Asking for What Nobody Labelled

🧠 **The intuition.** The model was trained on sentences containing "dark",
"bright", "narrow" and "wide". Those words were never labels, and no output neuron
corresponds to them, no loss term mentions them. They were simply present in the
text, and the text encoder had to make them mean something in order to tell
captions apart.

So: can we now *query* by them? Build 40 prompts covering every
class × shade × width combination, and for each test image pick the single best
one. Then score the three attributes separately. A supervised classifier cannot
be given this test at all.

💻 **The code.**

In [ ]:
combo, combo_key = [], []
for ci, c in enumerate(CLASS_NAMES):
    for si, s in enumerate(SHADE):
        for wi, w in enumerate(WIDTH):
            combo.append(f"a {s} {w} photo of a {c}")
            combo_key.append((ci, si, wi))

Zc = embed_texts(model, combo)
best = (Zi @ Zc.T).argmax(1).cpu().numpy()
key = np.array(combo_key)

acc_cls   = (key[best, 0] == yte).mean()
acc_shade = (key[best, 1] == s_te).mean()
acc_width = (key[best, 2] == w_te).mean()

print("one prompt chosen per image, out of 40; each attribute scored separately\n")
print(f"  class  : {acc_cls:.4f}   (10-way, chance 0.100)")
print(f"  shade  : {acc_shade:.4f}   ( 2-way, chance 0.500)  <- never a label")
print(f"  width  : {acc_width:.4f}   ( 2-way, chance 0.500)  <- never a label")

In [ ]:
# Retrieval: type a sentence, get the images closest to it.
queries = ["a dark narrow photo of a Trouser",
           "a bright wide photo of a Coat",
           "a dark wide photo of a Bag",
           "a bright narrow photo of a Sandal"]
Zq = embed_texts(model, queries)
top = (Zq @ Zi.T).topk(6, dim=1).indices.cpu().numpy()

fig, axes = plt.subplots(len(queries), 6, figsize=(9, 1.6 * len(queries)))
for r, q in enumerate(queries):
    for c, idx in enumerate(top[r]):
        axes[r, c].imshow(Xte[idx], cmap="gray", vmin=0, vmax=1)
        axes[r, c].axis("off")
    axes[r, 0].set_title(q, fontsize=8, loc="left")
fig.suptitle("Text-to-image retrieval: the six nearest images to each typed sentence", y=1.02)
plt.tight_layout()
plt.show()

**Both attribute scores land far above chance**, and neither was ever supplied
as a training target. The model learned "narrow" from the company that word
kept, appearing beside trousers and sandals and rarely beside bags, and
that is enough to make it a usable query term.

This is the property that a fixed label set structurally cannot have, and it is
why CLIP-style pretraining underpins so much of what came after: the text
encoder becomes a general-purpose way of *addressing* visual content. In
Chapter 12 the gap to Stable Diffusion 3 came down partly to "replace the class
embedding with a frozen text encoder", and this chapter is where that text
encoder comes from.


---
# Module 6 — Two Measurements That Complicate the Story

## 6.1 Does a bigger batch really help?

🧠 **The claim you will read everywhere.** Contrastive learning needs enormous
batches, because the batch *is* the negative set, and CLIP used 32,768. More
negatives, harder task, better representation.

📐 **Why it is hard to test.** Batch size does not move one thing, it moves
three: the number of negatives per example, the number of optimizer steps per
epoch, and the amount of gradient averaging. Hold epochs fixed and a bigger
batch gets **fewer updates**. Hold updates fixed and a bigger batch **sees more
data**. There is no setting in which only the negatives change.

So we measure both, and report both.

💻 **The code.** Six more training runs; this is the slow cell.


In [ ]:
def evaluate(m):
    Z = embed_images(m, img_te)
    T = embed_texts(m, prompts)
    return ((Z @ T.T).argmax(1).cpu().numpy() == yte).mean()


def train_steps(batch, steps, lr=3e-4, seed=0):
    '''Same loop, but stopped after a fixed number of optimizer updates.'''
    torch.manual_seed(seed)
    m = CLIP(tok.vocab_size).to(device)
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=0.1)
    ids, mask = enc_tr["input_ids"], enc_tr["attention_mask"]
    done = 0
    while done < steps:
        perm = torch.randperm(len(img_tr))
        for i in range(0, len(perm) - batch + 1, batch):
            j = perm[i:i+batch]
            loss = clip_loss(m(img_tr[j].to(device), ids[j].to(device), mask[j].to(device))[0])
            opt.zero_grad(); loss.backward(); opt.step()
            done += 1
            if done >= steps:
                break
    return m


BATCHES = (32, 128, 512)          # <- knob
print("A) fixed EPOCHS = 8  (equal data seen, unequal updates)")
fixed_epochs = {}
for b in BATCHES:
    fixed_epochs[b] = evaluate(train(batch=b, epochs=8, log=False)[0])
    print(f"   batch {b:>4} -> zero-shot {fixed_epochs[b]:.4f}   ({8 * N_TRAIN // b:>4} updates)")

print("\nB) fixed STEPS = 600  (equal updates, unequal data seen)")
fixed_steps = {}
for b in BATCHES:
    fixed_steps[b] = evaluate(train_steps(b, 600))
    print(f"   batch {b:>4} -> zero-shot {fixed_steps[b]:.4f}   ({b * 600 / 1000:>5.0f}k examples)")

### The two controls disagree, and that is the finding

At **fixed epochs**, accuracy *falls* as the batch grows. At **fixed optimizer
steps**, it *rises*. Same model, same data, opposite conclusions, because the
two protocols are asking different questions:

- Fixed epochs holds the data budget constant, so a larger batch is really
  "fewer, better-averaged updates". Fewer updates wins less.
- Fixed steps holds the update budget constant, so a larger batch is really
  "more negatives **and** more data". More of both wins more.

**Neither one isolates the negatives**, which is the quantity the folklore is
actually about. To do that you would need to decouple the loss's negative set
from the gradient batch, which is exactly what the methods invented for this
do (memory banks in MoCo, gathering embeddings across GPUs before computing the
loss, and CLIP's own sharded implementation).

The honest summary: our measurement is consistent with "big batches help", and
it does not demonstrate it. If you only ran protocol A you would have concluded
the opposite of the literature and been able to defend it with a plot.

## 6.2 The modality gap

🧠 **The intuition.** The loss pushed matching image and text embeddings
together, so you would expect a picture of a boot and the sentence describing it
to end up in roughly the same place on the sphere. They do not. Image embeddings
occupy one cone and text embeddings occupy a different, disjoint one, with the
matched pairs merely being *slightly* closer within that arrangement than the
mismatched ones.

📐 **What to measure** (Liang et al., 2022, *"Mind the Gap"*). Compare the mean
cosine similarity **within** each modality against the mean **across** them, and
the distance between the two centroids:

$$ \Delta = \big\lVert\, \overline{z_I} - \overline{z_T} \,\big\rVert_2 $$

If the two modalities shared a region of the sphere, $\Delta$ would be near zero
and the three similarity numbers would be comparable.

💻 **The code.**


In [ ]:
gi, gt = Zi.mean(0), Zt.mean(0)
gap = (gi - gt).norm().item()

sub = Zi[:1000]
print(f"mean cos(image, image) : {(sub @ sub.T).mean().item():+.4f}")
print(f"mean cos(text,  text)  : {(Zt @ Zt.T).mean().item():+.4f}")
print(f"mean cos(image, text)  : {(sub @ Zt.T).mean().item():+.4f}   <- the two modalities")
print(f"\ncentroid distance ||mean(img) - mean(txt)||: {gap:.4f}")

# Project both clouds onto the plane spanned by the two centroids.
basis = torch.linalg.qr(torch.stack([gi, gt], 1))[0]
pi, pt = (sub @ basis).cpu().numpy(), (Zt @ basis).cpu().numpy()

plt.figure(figsize=(6, 5))
plt.scatter(pi[:, 0], pi[:, 1], s=6, alpha=0.3, label="images")
plt.scatter(pt[:, 0], pt[:, 1], s=90, marker="*", label="class prompts")
plt.xlabel("centroid axis 1"); plt.ylabel("centroid axis 2")
plt.title("Two modalities, one space, two separate regions")
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**The gap is real and it reproduces here.** Cross-modal similarity is markedly
lower than similarity within either modality, and the two clouds sit in visibly
different places.

Why does zero-shot classification still work? Because the decision rule is an
$\arg\max$ **over prompts for a fixed image**, a comparison *within* one row of
the similarity matrix. A constant offset between the modalities shifts every
entry in that row by roughly the same amount and cancels out of the ranking. The
gap breaks absolute-similarity thresholds ("is this caption right for this
image, yes or no?"), not relative ones.

It is a good reminder of something this curriculum keeps running into: a model
can be extremely useful through a metric it does not actually optimize cleanly.
The same caution applied to attention weights as explanations in Chapter 9, and
to pixel distance as a diversity measure in Chapter 12.


---
# Wrap-Up

| You built | The transferable lesson |
|---|---|
| Image tower + text tower + two projections | Two unrelated encoders become comparable by ending both with a linear map into one L2-normalized space |
| The symmetric InfoNCE loss | Negatives are free: in a batch of $N$ pairs, $N^2 - N$ cells are wrong by construction. Cross-entropy against the diagonal, both directions |
| The learned temperature | Cosine is stuck in $[-1,1]$, so logits are too flat to train on; $1/\tau$ stretches them. The $\sqrt{d_k}$ argument from Chapter 9, run backwards |
| Zero-shot classification | The classifier head is a list of sentences: $\arg\max_c z_I \cdot z_{T_c}$ is a linear head whose weights you typed |
| Compositional prompts | Words that were never labels become usable query terms. This is what language buys over a fixed label set |
| The batch-size sweep | Two defensible protocols, opposite conclusions. Batch size moves negatives, updates, and data at once, and neither control isolates the first |
| The modality gap | Images and text occupy separate cones. Zero-shot survives it because $\arg\max$ within a row is invariant to a shared offset |

**What this unlocks.** A text encoder whose output vectors *address visual
content* is the missing piece in two places you have already seen:

- **Text-conditioned generation.** Chapter 12 ended by saying the gap to Stable
  Diffusion 3 was mostly "replace the class embedding with a frozen text
  encoder". This is that encoder. The null class becomes the empty string, and
  classifier-free guidance carries over untouched.
- **Feeding images to a language model.** Right now we can only *score* an
  image against a sentence. We cannot ask a question about an image and get a
  sentence back, because there is no decoder anywhere in this architecture:
  CLIP has two encoders and nothing that generates.

That second gap is the next chapter. A frozen image encoder like this one, a
frozen language model, and a small trained bridge between them turns out to be
enough, which is the LLaVA result, and it is a surprisingly cheap thing to
build.
